In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyprojroot import here
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [2]:
DATA_DIR = here("data/raw")
print(DATA_DIR)   # ...\SECOM\data\raw 나오는지 확인



c:\Users\Thinkpad\Desktop\python\SECOM\data\raw


In [3]:
# 피처 (공백 구분, 결측치는 NaN으로 들어있음)
X = pd.read_csv(DATA_DIR / "secom.data", sep=r"\s+", header=None)
X.columns = [f"f{i}" for i in range(X.shape[1])]

In [4]:
# 라벨 + 타임스탬프 (예: -1 "19/07/2008 11:55:00")
y = pd.read_csv(DATA_DIR / "secom_labels.data", sep=" ", header=None,
                names=["label", "timestamp"], quotechar='"')
y["timestamp"] = pd.to_datetime(y["timestamp"], format="%d/%m/%Y %H:%M:%S")

In [ ]:
X1 = X.loc[:, X.nunique(dropna=True) > 1]   # 상수 제거 (1개의 고유 값만 있는 열 제거)
yb = (y["label"] == 1).astype(int)             # 1 = Fail
print('raw_data: ', X.shape, y.shape)
print('X1_preprocessed: ',X1.shape,'불량품 수: ' ,yb.sum())

(1567, 474) 104


In [ ]:
print('X1_shape: ', X1.shape)
print('y_shape: ', y.shape)
print(f'양품 비율: {(y[y["label"] == -1].shape[0] / y.shape[0]):.4f}')  # 양품 비율 확인 # -1 pass/ 1 fail
print(f'불량품 비율: {(y[y["label"] == 1].shape[0] / y.shape[0]):.4f}')  # 불량품 비율 확인 # -1 pass/ 1 fail
print(f'양품 수: {y[y["label"] == -1].shape[0]}')
print(f'불량품 수: {y[y["label"] == 1].shape[0]}')

X_shape:  (1567, 474)
y_shape:  (1567, 2)
양품 비율: 0.9336
불량품 비율: 0.0664
양품 수: 1463
불량품 수: 104


In [ ]:
# ── 1) 피처 간 상관계수 행렬 ──────────────────────────────
# NaN이 있는 행은 쌍마다 알아서 제외하고 계산됨 (pairwise)
corr = X1.corr()   # NaN은 pairwise로 제외됨

# ── 2) 상관계수 → 거리로 변환 ─────────────────────────────
# |r| = 1  → 거리 0 (완전히 같이 움직임 = 중복)
# |r| = 0  → 거리 1 (서로 무관)
# abs를 쓰는 이유: r = -0.99처럼 반대로 움직여도 담고 있는 정보는 같음
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack()   # 상삼각행렬과 대각행렬 제거
high = pairs[pairs.abs() > 0.95].sort_values(key=abs, ascending=False)
print(len(high), "쌍이 |r| > 0.95")
high.head(20)

329 쌍이 |r| > 0.95


f492  f545    1.0
f358  f544    1.0
f209  f342    1.0
      f347    1.0
f342  f347    1.0
f579  f587    1.0
f74   f342    1.0
      f478    1.0
      f209    1.0
f580  f588    1.0
f581  f589    1.0
f578  f586    1.0
f85   f542    1.0
f220  f543    1.0
f74   f347    1.0
f209  f478    1.0
f206  f209    1.0
      f347    1.0
      f478    1.0
f347  f478    1.0
dtype: float64

In [29]:
high[high == 1.0].count()  # 상관계수 1인 쌍의 수 (완전히 중복)

np.int64(17)

In [ ]:
# fillna(0): 상관계수를 못 구한 쌍은 "무관"으로 취급
dist = (1 - corr.abs().fillna(0)).to_numpy()

# 자기 자신과의 거리는 0이어야 함 (계산 오차 방지용)
np.fill_diagonal(dist, 0)

# ── 3) 계층적 군집 ───────────────────────────────────────
# squareform: 정사각 거리행렬 → linkage가 받는 1차원 형태로 변환
# method="average": 두 그룹을 합칠지 판단할 때
#                   두 그룹 사이 모든 변수 쌍의 거리 "평균"을 사용
Z = linkage(squareform(dist, checks=False), method="average")

# ── 4) 거리 0.05에서 잘라 그룹 번호 부여 ──────────────────
# 거리 0.05 = 평균 |r| 0.95 → 평균 |r| ≥ 0.95인 변수끼리 같은 그룹
# t 값을 바꾸면 엄격함 조절 (0.02 = 더 엄격, 0.1 = 더 느슨)
groups = pd.Series(fcluster(Z, t=0.05, criterion="distance"), index=X.columns)

# 그룹별 변수 개수 (1 = 혼자인 변수, 2 이상 = 중복 그룹)
print(groups.value_counts().head(10))   # 크기 2 이상 = 중복 그룹

In [ ]:
# ── 5) 그룹마다 대표 변수 1개 선정 ────────────────────────
# 기준: 결측 비율이 가장 낮은 변수
# (라벨을 안 보는 기준이라 나중에 모델링에 써도 누수 걱정 없음)
rep = (pd.DataFrame({"group": groups, "na": X1.isna().mean()})
         .sort_values("na")          # 결측 적은 순으로 정렬
         .groupby("group").head(1)   # 그룹마다 맨 위(= 결측 최소) 1개
         .index)

X2 = X1[rep]
print(X1.shape, "→", X2.shape)

# ── 6) 확인용: 크기 2 이상인 그룹에 어떤 변수가 묶였는지 ────
# groups.map(groups.value_counts()): 각 변수가 속한 그룹의 크기
grp_map = groups[groups.map(groups.value_counts()) > 1].sort_values()
grp_map.head(30)